In [ ]:
# importo le librerie necessarie
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import matplotlib.colors as mcolors
from tabulate import tabulate
from colorama import Fore, Style
import plotly.express as px
import folium
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon
from sklearn.impute import SimpleImputer

%matplotlib inline

In [ ]:
# Data Selection
df_fatal = pd.read_csv('dataset.csv')
df_fatal

In [ ]:
# Data cleaning 

# Converto la colonna "date" in formato datetime
df_fatal['date'] = pd.to_datetime(df_fatal['date'])

In [ ]:
# cambio i nomi degli stati con il loro corrispettivo esteso

# Ottengo i valori unici nella colonna "state"
state_values = df_fatal['state'].unique()

# Dizionario delle sigle degli stati e corrispettivi estesi
state_mapping = {
    'WA': 'Washington','OR': 'Oregon','KS': 'Kansas','CA': 'California','CO': 'Colorado',
    'OK': 'Oklahoma','AZ': 'Arizona','IA': 'Iowa','PA': 'Pennsylvania','TX': 'Texas','OH': 'Ohio',
    'LA': 'Louisiana','MT': 'Montana','UT': 'Utah','AR': 'Arkansas','IL': 'Illinois','NV': 'Nevada',
    'NM': 'New Mexico','MN': 'Minnesota','MO': 'Missouri','VA': 'Virginia','NJ': 'New Jersey',
    'IN': 'Indiana','KY': 'Kentucky','MA': 'Massachusetts','NH': 'New Hampshire','FL': 'Florida',
    'ID': 'Idaho','MD': 'Maryland','NE': 'Nebraska','MI': 'Michigan','GA': 'Georgia','TN': 'Tennessee',
    'NC': 'North Carolina','AK': 'Alaska','NY': 'New York','ME': 'Maine','AL': 'Alabama',
    'MS': 'Mississippi','WI': 'Wisconsin','SC': 'South Carolina','DE': 'Delaware',
    'DC': 'District of Columbia','WV': 'West Virginia','HI': 'Hawaii','WY': 'Wyoming',
    'ND': 'North Dakota','CT': 'Connecticut','SD': 'South Dakota','VT': 'Vermont','RI': 'Rhode Island'
}

# Applico la mappatura alle sigle degli stati nel dataframe
df_fatal['state'] = df_fatal['state'].map(state_mapping)



In [ ]:
# Dizionario delle sigle delle razze e corrispettivi estesi
race_mapping = {
    'W': 'White',
    'B': 'Black',
    'H': 'Hispanic',
    'A': 'Asian',
    'N': 'Native American',
    'O': 'Other'
}

# Applico la mappatura alle sigle delle razze nel dataframe
df_fatal['race'] = df_fatal['race'].map(race_mapping)

# assegno undetermined ai valori nulli nella colonna race
df_fatal['race'] = df_fatal['race'].fillna('Undetermined')


In [ ]:
df_fatal.loc[df_fatal['signs_of_mental_illness'] == False, 'signs_of_mental_illness'] = 'No'
df_fatal.loc[df_fatal['signs_of_mental_illness'] == True, 'signs_of_mental_illness'] = 'Yes'

In [ ]:
df_fatal.loc[df_fatal['gender'] == 'M', 'gender'] = 'Male'
df_fatal.loc[df_fatal['gender'] == 'F', 'gender'] = 'Female'

In [ ]:
df_fatal.loc[df_fatal['body_camera'] == False, 'body_camera'] = 'No'
df_fatal.loc[df_fatal['body_camera'] == True, 'body_camera'] = 'Yes'

In [ ]:
# assegno undetermined ai valori nulli nella colonna race
df_fatal['armed'] = df_fatal['armed'].fillna('undetermined')

In [ ]:
df_fatal = df_fatal.drop('is_geocoding_exact', axis=1)

In [ ]:
df_fatal.isnull().sum()

In [ ]:
#Queste sono variabili categoriche e numeriche che potrebbero essere importanti per le tue analisi.
#Potresti considerare l’imputazione basata sulla moda (per le variabili categoriche) o sulla media/mediana (per le variabili numeriche).

# Crea un imputatore per le variabili numeriche
num_imputer = SimpleImputer(strategy='mean')

# Seleziona le colonne numeriche
num_cols = ['age']

# Applica l'imputazione alle colonne numeriche
df_fatal[num_cols] = num_imputer.fit_transform(df_fatal[num_cols])

In [ ]:
# Crea un imputatore per le variabili categoriche
cat_imputer = SimpleImputer(strategy='most_frequent')

# Seleziona le colonne categoriche
cat_cols = ['flee']

# Applica l'imputazione alle colonne categoriche
df_fatal[cat_cols] = cat_imputer.fit_transform(df_fatal[cat_cols])

In [ ]:
df_fatal.isnull().sum()

In [ ]:
df = df_fatal.dropna()
df.isnull().sum()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
duplicate_rows = df.duplicated(subset=['name', 'age']) 
df[duplicate_rows]

In [ ]:
df[df.name == 'Alexander Matthew Collins']

In [ ]:
df = df.drop(6445)

In [ ]:
df[df.name == 'Demilo Trayuon Hodge']

In [ ]:
df = df.drop(6318)

In [ ]:
df[df.name == 'Breen Reiss']

In [ ]:
df = df.drop(7897)

In [ ]:
df[df.name == 'Robert Hammitt']

In [ ]:
df = df.drop(7429)

In [ ]:
#non ci sono più duplicati
duplicate_rows = df.duplicated(subset=['name', 'age']) 
df[duplicate_rows]

In [ ]:
df

OMICIDI NEL TEMPO

In [ ]:
# Data exploration

# calcolo la data minima e massima nel dataframe (più vecchia e più recente)
min_date = df['date'].min()
max_date = df['date'].max()

# calcolo il conteggio totale degli omicidi registrati
total_kill = len(df)

# divido gli omicidi per anno 

# estraggo l'anno dalla colonna 'date' e calcolo il conteggio per ogni anno
kill_by_year = df['date'].dt.year.value_counts().sort_index()

# calcolo la percentuale per ogni anno
percentages = kill_by_year / total_kill * 100

# stampo
print(f'Data inizio: {min_date}\nData fine: {max_date}\n')
print(f'Totale omicidi registrati: {total_kill}\n')

# Trova l'anno con il conteggio maggiore
anno_max = kill_by_year.idxmax()

# Prepara i dati per la tabella
table_data = []
for year, count in kill_by_year.items():
    percentage = percentages[year]
    if year == anno_max:
        table_data.append([Fore.RED + str(year), count, f'{percentage:.2f}%' + Style.RESET_ALL])
    else:
        table_data.append([year, count, f'{percentage:.2f}%'])

# Stampa la tabella
headers = ['Anno', 'Conteggio', 'Percentuale']
print(tabulate(table_data, headers, tablefmt='fancy_grid'))

Scopriamo quindi che i dati si riferiscono al periodo dal 2 gennaio 2015 al 29 novembre 2022.
Il totale degli omicidi da parte della polizia negli Stati Uniti in questo lasso di tempo è di 6794 persone.
La tabella mostra il conteggio degli omicidi per ogni anno, insieme alla percentuale che rappresenta rispetto al totale. In rosso faccio risaltare l'anno con il numero più alto di omicidi, che risulta essere il 2015 che con 9936 vittime rappresenta il 14.62% del totale delle vittime di questo dataset

In [ ]:
# Data transformation

# Creazione del nuovo dataframe
df_kill_by_year = pd.DataFrame({
    'year': kill_by_year.index,
    'total_count': kill_by_year.values,
    'percent': percentages.values.round(2)
})

# calcolo la media annuale
average_kill_by_year = df_kill_by_year['total_count'].mean()

# estraggo l'anno e il mese dalla colonna 'date'
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month

# calcolo il conteggio degli omicidi per ogni combinazione di anno e mese
total_kill_by_year_month = df.groupby(['year', 'month']).size().reset_index(name='count')

# creo un altro DataFrame con anni e mesi come colonne
df_kill_by_year_month = pd.pivot_table(total_kill_by_year_month, values='count', index='year', columns='month')

# conversione in interi
df_kill_by_year_month = df_kill_by_year_month.astype('Int64')

# raggruppo i dati per anno e razza e ne faccio un conteggio
race_count = df.groupby(['year', 'race']).size().reset_index(name='count')

# collego il dataframe con quello del conteggio annuale
df_race_merged = race_count.merge(df_kill_by_year, on='year')


In [ ]:
# Data visualization

# imposto stile seaborn 
sns.set_style('darkgrid')

# creazione del grafico a linee
plt.figure(figsize=(10,6))
plt.plot(df_kill_by_year['year'].values, df_kill_by_year['total_count'].values, marker='o', linestyle='-', color='red', markersize=6)

# aggiunta della linea tratteggiata per la media annuale
plt.axhline(y=average_kill_by_year, color='red', linestyle=':', label='Media annuale')

# aggiunta dell'etichetta della media annuale
plt.text(df_kill_by_year['year'].iloc[-1] + 0.4, average_kill_by_year+4, f'{int(average_kill_by_year)}\n Media annuale', ha='left', va='top')

# aggiunta delle etichette ad ogni punto
for i, j in zip(df_kill_by_year['year'], df_kill_by_year['total_count']):
    plt.text(i, j+5, str(j), ha='center', va='bottom')

# titoli degli assi e del grafico
plt.xlabel('Anno', fontweight='bold')
plt.ylabel('Numero omicidi', fontweight='bold')
plt.title('Distribuzione degli omicidi negli anni', fontsize=20,fontweight='bold')

# visualizzazione del grafico
plt.show()

Con questo grafico vediamo chiaramente che dal 2015 al 2020 c'è una costante tendenza, anche se minima, al decremento del numero degli omicidi da parte della polizia. Nel triennio 2020 - 2022 si è verificato invece un notevole decremento. La media annuale dei morti è 849, l'anno con il numero più basso è il 2022, tenendo conto che nel mese di dicembre non ci sono stati omicidi.

In [ ]:
# faccio il grafico
plt.figure(figsize=(10, 6))
custom_palette = ['yellow', 'black', 'orange', 'red', 'green', 'purple', 'pink']
sns.lineplot(data=df_race_merged, x='year', y='count', hue='race', marker='o',
            palette=custom_palette)

# titolo, legenda, assi
plt.title('Distribuzione degli omicidi per etnia negli anni', fontsize=20, fontweight='bold')
plt.legend(title='Etnia', loc='upper left', facecolor='lightgrey', bbox_to_anchor=(1, 1))
plt.xlabel('Anno', fontweight='bold')
plt.ylabel('Numero di omicidi', fontweight='bold')

# visualizzo il grafico
plt.show()

Questo grafico rappresenta la distribuzione degli omicidi per etnia nel corso del tempo.
L'andamento è abbastanza lineare per alcune etnie.
In generale, per l'etnia White, Black e Hispanic, le più colpite, sembra esserci la tendenza ad una diminuzione dei casi nel tempo.
I dati l'etnia che ho aggiunto, 'undetermined', hanno una tendenza, invece, ad aumentare nel tempo. Soprattutto dal 2020, il numero di vittime di cui non si ha informazione sulla razza sono in aumento vertiginoso; questo può essere dovuto ad una molteplicità di fattori, il cui studio esula però dagli obiettivi di questa analisi. Il grafico qua sotto rafforza le idee qua espresse.

In [ ]:
# Creo il grafico a barre 
plt.figure(figsize=(10, 6))

sns.barplot(data=df_race_merged, x='year', y='count', hue='race', 
            palette=custom_palette, 
            edgecolor='black', 
            linewidth=2, 
            saturation=1, 
            alpha=0.6)

# Titolo, legenda, assi
plt.title('Numero omicidi per etnia negli anni', fontsize=20, fontweight='bold')
plt.legend(title='Etnia', loc='upper left', facecolor='white', bbox_to_anchor=(1, 0.9))
plt.xlabel('Anno', fontweight='bold')
plt.ylabel('Numero di omicidi', fontweight='bold')

# Visualizzo il grafico
plt.show()

Con questo grafico riusciamo a vedere le informazioni adatte per confrontare il numero di vittime di ogni etnia in ogni anno e conferma i dati analizzati finora.

ANALISI DI ETA', GENERE, ETNIA

In [ ]:
# age stats
df['age'].describe().round(2)

Ll conteggio che si delinea è di 6794 vittime. L'età media è di circa 37 anni, la vittima più giovane ne aveva appena 2, e la più anziana 91.

In [ ]:
# stats per genere
df.groupby('gender')['age'].describe().round(2)

In [ ]:
# Calcolo la percentuale rispetto al totale
gender_counts = df['gender'].value_counts()
total_count = gender_counts.sum()
gender_percentages = gender_counts / total_count * 100

# Formatto le percentuali come stringhe con il simbolo '%'
formatted_percentages = gender_percentages.map('{:.2f}%'.format)
formatted_percentages

Le età medie sono simili così come quelle massime. C'è una differenza nelle età minime, dove quelle femminile segna 2 anni, e maschile 6. Più del 95% delle vittime di cui sappiamo l'età sono vittime di sesso maschile; il sesso femminile rappresenta solo il 4.68% dei casi.

In [ ]:
# creo histplot
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='age', hue='gender', kde=True, multiple='stack')


plt.xlabel('Età', fontweight='bold')
plt.ylabel('Frequenza', fontweight='bold')
plt.title('Distribuzione delle età per genere', fontsize=20, fontweight='bold')

plt.show()

Con questo istogramma studiamo la distribuzione delle età per genere. Come ci aspettiamo di trovare, vediamo che il grafico ha colonne molto alte per il genere maschile, e soprattutto nella fascia d'età 20-40, come sembra suggerire anche la linea blu che segue la distribuzione e che trova il suo picco proprio lì.

In [ ]:
# creao histplot per età e razza
plt.figure(figsize=(12, 8))
race_palette = ['yellow', 'pink', 'orange', 'grey', 'green', 'purple', 'red']
sns.histplot(data=df, x='age', hue='race', multiple='stack', palette=race_palette, edgecolor='black')

plt.title('Distribuzione delle età per razza', fontsize=15, fontweight='bold')
plt.xlabel('Età', fontweight='bold')
plt.ylabel('Totale casi', fontweight='bold')

plt.show()


Questo grafico prende in considerazione le età e le razze. Come si puà notare le barre sono impilate, mostrando la quantità effettiva che rappresenta quella razza in quella precisa fascia d'età, usando colori differenti come esplicitato nella legenda. Si riescono a notare alcuni trend, come il range d'età coperto dai Native American ad esempio, che risulta essere più corto rispetto alle altre etnie. In generale si può anche notare, tra le altre cose, che nelle fasce d'età molto giovani sono presenti in modo significativo i colori rappresentanti soprattutto le etnie Hispanic e Black.

In [ ]:
# stats 
age_race = df.groupby("race")["age"].agg(["count", "min", "max", "mean"]).round(2)
age_race

Questo output ci da un' idea dettagliata della distribuzione delle età delle vittime, dividendole per razza. La media delle età è tutto sommato abbastanza variabile, con la media più basse che spetta ai Native American(32.5) seguiti dai Black(32.9), la più alta invece a Undetermined(40.3) e White(40). Per ogni etnia abbiamo quindi il conteggio totale, l'età minima, massima, e la media delle età.

In [ ]:
# conteggio e percentuale per razza
race_count = df['race'].value_counts()
race_percentage = race_count / total_count * 100

In [ ]:
# Crea una lista di colori mappati alle etnie desiderate
ethnicity_colors = {'White': 'pink', 'Black': 'grey', 'Hispanic': 'orange', 'Asian': 'yellow', 'Native American': 'red', 'Other': 'green', 'Undetermined': 'purple'}
sorted_palette = [ethnicity_colors[ethnicity] for ethnicity in race_percentage.index]

# Crea una lista di esplosioni per le ultime tre fette
explode = [0] * (len(race_count) - 3) + [0.1, 0.4, 0.6]

# Crea il grafico a torta
plt.figure(figsize=(8, 8))
wedges, _, _ = plt.pie(race_percentage, labels=[''] * len(race_percentage), autopct='%.2f%%', colors=sorted_palette, explode=explode, wedgeprops={'edgecolor': 'black'}, pctdistance=1.1)

# Rimuovi etichetta dell'asse y
plt.ylabel('')

# Aggiungi la legenda con i colori delle etnie
plt.legend(wedges, race_percentage.index, loc='upper left', bbox_to_anchor=(1, 0.9), title='Etnia', facecolor='lightgrey')

# Titolo del grafico
plt.title('Composizione etnica degli omicidi', fontsize=20, fontweight='bold')

# Visualizza il grafico
plt.show()


Questo grafico a torta rappresenta la composizione etnica degli omicidi negli Stati Uniti nel periodo preso in considerazione. E' interessante notare che ci sono alcune etnie che presentano la maggior parte delle vittime, come l'etnia 'White'(44%) che da sola conta quasi la metà delle vittime totali. Più del 23% sono di etnia 'Black' ed esattamente il 16% Hispanic. Le restanti etnie rappresentano una percentuale molto piccola, Asian 1.77% e Native American 1.35%. Abbiamo poi una 'fetta' della torta di colore viola, che rappresenta le vittime di cui non si hanno informazioni sulla razza, ovvero il 12.56%.

ANALISI GEOGRAFICA

In [ ]:
# Data exploration/transformation
# conteggio per stato
state_count = df['state'].value_counts()

# creo nuovo DataFrame con il conteggio totale per ogni stato
df_state = pd.DataFrame({'state': state_count.index, 'count': state_count.values})

# aggiungo una colonna percent
df_state['percent'] = ((df_state['count'] / total_kill) * 100).round(2)

df_state.head(10)

Nell'output ci sono i 10 stati 'peggiori'. Ho creato questo dataframe contenente dati suddivisi per stato; il totale delle vittime e la percentuale che rappresenta rispetto al totale delle vittime della polizia negli Stati Uniti.

In [ ]:
# definisco la scala di colori per le barre
cmap = mcolors.LinearSegmentedColormap.from_list('custom_colors', ['#008000', '#FFA500', '#FF0000'])

# creo il grafico a barre orizzontali
plt.figure(figsize=(10, 12))
plt.barh(df_state['state'], df_state['count'], color=cmap(df_state['count']), edgecolor='black')

# Inverti l'ordine delle barre
plt.gca().invert_yaxis()

# assi e titolo
plt.xlabel('Totale vittime', fontweight='bold')
plt.ylabel('Stato', fontweight='bold')
plt.title('Omicidi per stato', fontsize=20, fontweight='bold')

plt.show()

Questo grafico a barre orizzontali serve a confrontare visivamente il totale delle vittime per ogni stato. Ho impostato una gamma di 3 colori che aiutano a comprendere meglio i dati e suddividere in 3 grandi categorie visive gli stati. Non solo i colori, ma anche l'ordine degli stati aiuta a comprendere immediatamente e meglio i dati. La California è il posto dove si verifica il numero più alto di omicidi da parte della polizia, con quasi 950 vittime. Segue in classifica il Texas, poi Florida, Arizona e Georgia completano questa triste top five.

In [ ]:
# creazione dataframe per stato con i numeri delle etnie
ethn = df.groupby(['state', 'race']).size().unstack().reset_index()

# unione con il dataset con conteggio totale per stato e percentuale
df_state_stats = pd.merge(df_state, ethn, on='state')

# Sostituisco i valori NaN con 0
df_state_stats.fillna(0, inplace=True)

# sposto count e percent a destra per mia comodità di lettura
df_state_stats = df_state_stats[[col for col in df_state_stats.columns if col not in ['count', 'percent']] + ['count', 'percent']]
df_state_stats

In [ ]:
fig = px.density_mapbox(df, lat='latitude', lon='longitude', radius=5,
                        center=dict(lat=37, lon=-95), zoom=3,
                        mapbox_style="stamen-terrain")

fig.show()

Ho unito il dataframe con il conteggio per stato e la percentuale a quello appena creato con il conteggio di ogni gruppo etnico suddiviso per stato. Il risultante è questo qua sopra in cui ci sono informazioni dettagliate per ogni stato: abbiamo il numero di morti di ogni etnia, il numero di morti di cui non si sa l'etnia, il numero totale dei morti nello stato, e la percentuale rispetto al totale degli Usa.

In [ ]:
# carico i dati geografici degli stati uniti utilizzando geopandas
us_states = gpd.read_file('https://raw.githubusercontent.com/python-visualization/folium/master/examples/data/us-states.json')

# conteggio totale numero morti per stato
state_counts = df.groupby('state').size().reset_index(name='count')

# unisco dati degli stati con conteggio dei morti
merged_data = us_states.merge(state_counts, left_on='name', right_on='state', how='left')

# creo mappa centrata sulle coordinate degli Stati Uniti
m = folium.Map(location=[37.0902, -95.7129], zoom_start=4)

# Aggiungo un layer Choropleth per visualizzare il conteggio totale dei morti per stato
folium.Choropleth(
    geo_data=merged_data,
    name='Conteggio Morti',
    data=merged_data,
    columns=['name', 'count'],
    key_on='feature.properties.name',
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Conteggio totale di omicidi'
).add_to(m)

# Aggiungo il testo di popup con il conteggio totale dei morti al passaggio del mouse su uno stato
for _, row in merged_data.iterrows():
    state_name = row['name']
    state_count = row['count']
    geometry = row['geometry']
    
    # Gestisci sia i poligoni singoli che i poligoni multipli
    if isinstance(geometry, Polygon):
        centroid = geometry.centroid
    elif isinstance(geometry, MultiPolygon):
        centroid = geometry.representative_point()
    else:
        continue
    
    text = f"{state_name}<br>{state_count}"
    folium.Marker(
        location=[centroid.y, centroid.x],
        icon=folium.Icon(icon='info-sign'),
        popup=folium.Popup(text)
    ).add_to(m)

# Mostra la mappa
m

Questa mappa degli Stati Uniti risulta più completa della precedente e abbiamo molte più informazioni che possiamo leggere, anche interattivamente. Per esempio questa mappa è colorata in base ad un scala di colori dal giallo al rosso ad indicare il numero di omicidi in quello stato. Si vede immediatamente a colpo d'occhio la differenza di numero di morti tra gli stati: in più, ho inserito una sorta di 'punto informativo' al centro di ogni stato. Con un semplice click su di esso, avremmo accesso a informazioni più dettagliate come il nome dello stato, e il numero totale di morti di quello stato.

In [ ]:
df

In [ ]:
# Specificare le colonne da eliminare
columns_to_drop = ['date', 'year', 'month']

# Eliminare le colonne specificate
data = df.drop(columns=columns_to_drop)

In [ ]:
data.head()

CLUSTERING

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [ ]:
columns_to_drop = ['name', 'id', 'city', 'state', 'gender', 'body_camera', 'signs_of_mental_illness']

# Creazione di un nuovo DataFrame 'cluster_data' eliminando le colonne specificate
cluster_data = data.drop(columns=columns_to_drop)


In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_cols = ['age', 'longitude', 'latitude']
scaler = StandardScaler()
cluster_data[numeric_cols] = scaler.fit_transform(cluster_data[numeric_cols])

cluster_data = pd.get_dummies(data=cluster_data, columns=['manner_of_death', 'armed','race', 'threat_level', 'flee'])



In [ ]:
cluster_data.head()

In [ ]:
from sklearn.cluster import KMeans
k_values = range(2, 11)
wcss_values = []
for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=24, max_iter=50, n_init=10)
    kmeans.fit(cluster_data)
    wcss = kmeans.inertia_
    wcss_values.append(wcss)
plt.plot(k_values, wcss_values, marker='o')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('WCSS')
plt.title('Elbow Method')
plt.show()

In [ ]:
from sklearn.cluster import KMeans
num_clusters = 5
kmeans = KMeans(n_clusters=num_clusters, random_state=25)
kmeans.fit(cluster_data)
cluster_labels = kmeans.labels_
cluster_data['Cluster'] = cluster_labels
print(cluster_data['Cluster'])

In [ ]:
from sklearn.metrics import silhouette_score
silhouette_scores = silhouette_score(cluster_data, cluster_labels)
print("Average Silhouette Score:", silhouette_scores)

In [ ]:
from sklearn.decomposition import PCA
# Initialize the PCA model
pca = PCA()
# Fit the model to the data
pca.fit(cluster_data)
# Transform the data using PCA
scaled_cluster_data_pca = pca.transform(cluster_data)
# Print the explained variance ratio
print("Explained Variance Ratio:", pca.explained_variance_ratio_)

In [ ]:
# Create a scatter plot of the PCA results with cluster labels
scatter = plt.scatter(scaled_cluster_data_pca[:, 0], scaled_cluster_data_pca[:, 1], c=cluster_labels, cmap='viridis')

# Aggiungi la legenda
legend_labels = ['Cluster {}'.format(i) for i in range(len(set(cluster_labels)))]
plt.legend(handles=scatter.legend_elements()[0], labels=legend_labels)

# Add labels and title to the plot
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('PCA Results with Clusters')
# Show the plot
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd


In [ ]:
# Creare la figura e l'asse 3D
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

# Creare lo scatter plot 3D
scatter = ax.scatter(scaled_cluster_data_pca[:, 0], scaled_cluster_data_pca[:, 1], scaled_cluster_data_pca[:, 2], c=cluster_labels, cmap='viridis')

# Aggiungere la legenda
legend_labels = ['Cluster {}'.format(i) for i in range(len(set(cluster_labels)))]
ax.legend(handles=scatter.legend_elements()[0], labels=legend_labels)

# Aggiungere etichette agli assi e un titolo
ax.set_xlabel('Principal Component 1')
ax.set_ylabel('Principal Component 2')
ax.set_zlabel('Principal Component 3')
ax.set_title('PCA Results with Clusters')

# Mostrare il grafico
plt.show()


In [ ]:
from collections import Counter

# Contare i punti per ciascun cluster
counts = Counter(cluster_labels)

# Visualizzare i risultati
for cluster, count in counts.items():
    print(f"Cluster {cluster}: {count} punti")


In [ ]:
df['Cluster'] = cluster_labels
df.head()

In [ ]:
# print 5 vitims from each cluster
from tabulate import tabulate

def similar(i):
    similar = df.loc[df['Cluster'] == i][:5].reset_index()
    similar_print = similar[["name", "race", "manner_of_death", "age", "gender", "signs_of_mental_illness", "flee", "armed", "state"]]
    print("\nCluster ", i+1, "victims:")
    print(tabulate(similar_print, headers='keys', tablefmt='psql'))

for i in range(5):
    similar(i)

In [ ]:
data['Cluster'] = cluster_labels
data.head()

In [ ]:
cluster_data.head()

In [ ]:
# Creiamo i boxplot per ogni feature separatamente per ciascun cluster

plt.figure(figsize=(10, 6))
sns.boxplot(x='Cluster', y=data['age'], data=data, hue='Cluster', palette='viridis')
plt.title(f'Boxplot of age by Cluster')
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Creare una figura
plt.figure(figsize=(15, 6))

# Primo grafico
plt.subplot(2, 2, 1)
sns.countplot(x='threat_level', hue='Cluster', data=df, palette='viridis')
plt.title('Distribuzione del livello di minaccia per i cluster')

# Secondo grafico
plt.subplot(2, 2, 2)
sns.countplot(x='race', hue='Cluster', data=df, palette='viridis')
plt.title('Distribuzione dell etnia per i cluster')

# Terzo grafico
plt.subplot(2, 2, 3)
sns.countplot(x='flee', hue='Cluster', data=df, palette='viridis')
plt.title('Distribuzione della fuga per i cluster')

# Quarto grafico
plt.subplot(2, 2, 4)
sns.boxplot(x='Cluster', y=data['age'], data=df, hue='Cluster', palette='viridis')
plt.title(f'Boxplot dell età per i cluster')


# Mostrare i grafici
plt.tight_layout()
plt.show()




In [ ]:
sampled_data = df.sample(n=500, random_state=0)
# creo histplot
plt.figure(figsize=(30, 6))
sns.countplot(x='armed', hue='Cluster', data=sampled_data, palette='viridis')
plt.title('Distribuzione di "flee" per i cluster')

plt.show()

associations rules

In [ ]:
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

In [ ]:
data.head()

In [ ]:
#Filter the transactions for cluster 0
cluster_0 = cluster_data[cluster_data['Cluster'] == 0].drop(['Cluster', 'age', 'latitude', 'longitude'],axis=1)
# Generate frequent itemsets using Apriori algorithm
frequent_itemsets_0 = apriori(cluster_0, min_support=0.015,use_colnames=True)
# Generate association rules
rules_0 = association_rules(frequent_itemsets_0, metric="lift", min_threshold=1)
# Sort the rules by lift in descending order
rules_0 = rules_0.sort_values(by='lift', ascending=False)

# Print the top 10 association rules
rules_0.head(10)

In [ ]:
frequent_itemsets_0.head(10).sort_values(by='support', ascending=False)

In [ ]:
r0 = rules_0[ (rules_0['lift'] >= 5) & (rules_0['confidence'] >= 0.5) ]
r0

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Generate scatterplot using support and confidence
sns.scatterplot(x = "support", y = "confidence",size = "lift", data = r0)
plt.show()

In [ ]:
#Filter the transactions for cluster 1
cluster_1 = cluster_data[cluster_data['Cluster'] == 1].drop(['Cluster', 'age', 'latitude', 'longitude'],axis=1)
# Generate frequent itemsets using Apriori algorithm
frequent_itemsets_1 = apriori(cluster_1, min_support=0.015,use_colnames=True)
# Generate association rules
rules_1 = association_rules(frequent_itemsets_1, metric="lift", min_threshold=1)
# Sort the rules by lift in descending order
rules_1 = rules_1.sort_values(by='lift', ascending=False)

# Print the top 10 association rules
rules_1.head(10)

In [ ]:
frequent_itemsets_1.head(10).sort_values(by='support', ascending=False)

In [ ]:
r1 = rules_1[ (rules_1['lift'] >= 4) & (rules_1['confidence'] >= 0.5) ]
r1

In [ ]:
# Generate scatterplot using support and confidence
sns.scatterplot(x = "support", y = "confidence",size = "lift", data = r1)
plt.show()

In [ ]:
#Filter the transactions for cluster 2
cluster_2 = cluster_data[cluster_data['Cluster'] == 2].drop(['Cluster', 'age', 'latitude', 'longitude'],axis=1)
# Generate frequent itemsets using Apriori algorithm
frequent_itemsets_2 = apriori(cluster_2, min_support=0.015,use_colnames=True)
# Generate association rules
rules_2 = association_rules(frequent_itemsets_2, metric="lift", min_threshold=1)
# Sort the rules by lift in descending order
rules_2 = rules_2.sort_values(by='lift', ascending=False)

# Print the top 10 association rules
rules_2.head(10)

In [ ]:
frequent_itemsets_2.head(10).sort_values(by='support', ascending=False)

In [ ]:
r2 = rules_2[ (rules_2['lift'] >= 5) & (rules_2['confidence'] >= 0.5) ]
r2

In [ ]:
sns.scatterplot(x = "support", y = "confidence",size = "lift", data = r2)
plt.show()

In [ ]:
cluster_3 = cluster_data[cluster_data['Cluster'] == 3].drop(['Cluster', 'age', 'latitude', 'longitude'],axis=1)
# Generate frequent itemsets using Apriori algorithm
frequent_itemsets_3 = apriori(cluster_3, min_support=0.015,use_colnames=True)
# Generate association rules
rules_3 = association_rules(frequent_itemsets_3, metric="lift", min_threshold=1)
# Sort the rules by lift in descending order
rules_3 = rules_3.sort_values(by='lift', ascending=False)
rules_3.head(10)

In [ ]:
frequent_itemsets_3.head(10).sort_values(by='support', ascending=False)

In [ ]:
r3 = rules_3[ (rules_3['lift'] >= 5) & (rules_3['confidence'] >= 0.7) ]
r3

In [ ]:
sns.scatterplot(x = "support", y = "confidence",size = "lift", data = r3)
plt.show()

In [ ]:
cluster_4 = cluster_data[cluster_data['Cluster'] == 4].drop(['Cluster', 'age', 'latitude', 'longitude'],axis=1)
# Generate frequent itemsets using Apriori algorithm
frequent_itemsets_4 = apriori(cluster_4, min_support=0.015,use_colnames=True)
# Generate association rules
rules_4 = association_rules(frequent_itemsets_4, metric="lift", min_threshold=1)
# Sort the rules by lift in descending order
rules_4 = rules_4.sort_values(by='lift', ascending=False)
rules_4.head(10)

In [ ]:
frequent_itemsets_4.head(10).sort_values(by='support', ascending=False)

In [ ]:
r4 = rules_4[ (rules_4['lift'] >= 5) & (rules_4['confidence'] >= 0.7) ]
r4

In [ ]:
sns.scatterplot(x = "support", y = "confidence",size = "lift", data = r4)
plt.show()

In [ ]:
cluster_data.drop('Cluster', axis=1, inplace=True)
cluster_data.head()

In [ ]:
cluster_data.head()

In [ ]:
data.drop('Cluster', axis=1, inplace=True)

In [ ]:
columns_to_drop = ['name', 'id', 'longitude', 'latitude']

# Creazione di un nuovo DataFrame 'cluster_data' eliminando le colonne specificate
pattern_data = data.drop(columns=columns_to_drop)


In [ ]:
pattern_data.drop('age', axis=1, inplace=True)

In [ ]:
pattern_data.head()

In [ ]:


##convert categorial variables to dummy vars to use kmeans algotithm 
pattern_data = pd.get_dummies(data=pattern_data, columns=['manner_of_death', 'armed','race', 
                                   'threat_level', 'flee', 'state', 'city', 'signs_of_mental_illness', 'body_camera', 'gender'])


In [ ]:
pattern_data.head()

In [ ]:

# Generate frequent itemsets using Apriori algorithm
frequent_itemsets = apriori(pattern_data, min_support=0.015,use_colnames=True)
# Generate association rules
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)
# Sort the rules by lift in descending order
rules = rules.sort_values(by='lift', ascending=False)
rules.head(10)

In [ ]:
frequent_itemsets.head(10).sort_values(by='support', ascending=False)

In [ ]:
data.head()

In [ ]:
columns_to_drop = ['name', 'id', 'longitude', 'latitude']

# Creazione di un nuovo DataFrame 'cluster_data' eliminando le colonne specificate
pattern_data = data.drop(columns=columns_to_drop)

In [ ]:

bins = [0, 2, 18, 35, 65, np.inf]
names = ['<2', '2-18', '18-35', '35-65', '65+']

pattern_data['AgeRange'] = pd.cut(pattern_data['age'], bins, labels=names)
pattern_data.head()

In [ ]:
#convert to object type
pattern_data.AgeRange = pattern_data.AgeRange.astype('object')

#remove age columns
pattern_data.drop('age', axis = 1, inplace = True)

In [ ]:
pattern_data.head()

In [ ]:
##convert categorial variables to dummy vars to use kmeans algotithm 
pattern_data = pd.get_dummies(data=pattern_data, columns=['manner_of_death', 'armed','race', 
                                   'threat_level', 'flee', 'state', 'city', 'signs_of_mental_illness', 'body_camera', 'gender', 'AgeRange'])


In [ ]:

frequent_itemsets = apriori(pattern_data, min_support=0.015,use_colnames=True)

rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)

rules = rules.sort_values(by='lift', ascending=False)
rules.head(10)

In [ ]:
r = rules[ (rules['lift'] >= 5) & (rules['confidence'] >= 0.4) ]
r

In [ ]:
columns_to_drop = ['name', 'id', 'city', 'state', 'longitude', 'latitude']

# Creazione di un nuovo DataFrame 'cluster_data' eliminando le colonne specificate
cluster_data = data.drop(columns=columns_to_drop)


In [ ]:
#create categorical category for age 
bins = [0, 2, 18, 35, 65, np.inf]
names = ['<2', '2-18', '18-35', '35-65', '65+']

cluster_data['AgeRange'] = pd.cut(cluster_data['age'], bins, labels=names)
cluster_data.head()

In [ ]:
#convert to object type
cluster_data.AgeRange = cluster_data.AgeRange.astype('object')

#remove age columns
cluster_data.drop('age', axis = 1, inplace = True)

In [ ]:
cluster_data.dtypes

In [ ]:
#looking at number 
armed_counts = pd.DataFrame(cluster_data.AgeRange.value_counts()).head(15).reset_index()
armed_counts



In [ ]:
##convert categorial variables to dummy vars to use kmeans algotithm 
scaled_cluster_data = pd.get_dummies(data=cluster_data, columns=['manner_of_death', 'armed', 'gender', 'race', 'signs_of_mental_illness', 
                                   'threat_level', 'flee', 'body_camera', 'AgeRange'])

In [ ]:
scaled_cluster_data.head()

In [ ]:
scaled_cluster_data = scaled_cluster_data.astype('int64')
scaled_cluster_data.head()

In [ ]:
from sklearn.decomposition import PCA
# Initialize the PCA model
pca = PCA()
# Fit the model to the data
pca.fit(scaled_cluster_data)
# Transform the data using PCA
scaled_cluster_data_pca = pca.transform(scaled_cluster_data)
# Print the explained variance ratio
print("Explained Variance Ratio:", pca.explained_variance_ratio_)

In [ ]:
print("Explained Variance Ratio:", pca.explained_variance_ratio_)
# Visualizza l'importanza delle variabili originali nelle prime componenti principali
plt.figure(figsize=(8, 6))
plt.bar(range(len(pca.components_)), pca.explained_variance_ratio_)
plt.xlabel('Componente principale')
plt.ylabel('Varianza spiegata')
plt.title('Varianza spiegata da ciascuna componente principale')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
plt.scatter(scaled_cluster_data_pca[:, 0], scaled_cluster_data_pca[:, 1])
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('PCA Results')
plt.show()
12

In [ ]:
from sklearn.cluster import KMeans
k_values = range(2, 11)
wcss_values = []
for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=24, max_iter=50, n_init=10)
    kmeans.fit(scaled_cluster_data)
    wcss = kmeans.inertia_
    wcss_values.append(wcss)
plt.plot(k_values, wcss_values, marker='o')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('WCSS')
plt.title('Elbow Method')
plt.show()

In [ ]:
from sklearn.cluster import KMeans
num_clusters = 3
kmeans = KMeans(n_clusters=num_clusters, random_state=25)
kmeans.fit(scaled_cluster_data)
cluster_labels = kmeans.labels_
scaled_cluster_data['Cluster'] = cluster_labels
print(scaled_cluster_data['Cluster'])

In [ ]:
from sklearn.metrics import silhouette_score
silhouette_scores = silhouette_score(scaled_cluster_data, cluster_labels)
print("Average Silhouette Score:", silhouette_scores)

In [ ]:
# Create a scatter plot of the PCA results with cluster labels
scatter = plt.scatter(scaled_cluster_data_pca[:, 0], scaled_cluster_data_pca[:, 1], c=cluster_labels, cmap='viridis')

# Aggiungi la legenda
legend_labels = ['Cluster {}'.format(i) for i in range(len(set(cluster_labels)))]
plt.legend(handles=scatter.legend_elements()[0], labels=legend_labels)

# Add labels and title to the plot
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('PCA Results with Clusters')
# Show the plot
plt.show()

In [ ]:
cluster_0 = scaled_cluster_data[scaled_cluster_data['Cluster'] == 0].sum().sort_values(ascending=False)
print(cluster_0.head(10))

In [ ]:
cluster_1 = scaled_cluster_data[scaled_cluster_data['Cluster'] == 1].sum().sort_values(ascending=False)
print(cluster_1.head(10))

In [ ]:
cluster_2 = scaled_cluster_data[scaled_cluster_data['Cluster'] == 2].sum().sort_values(ascending=False)
print(cluster_2.head(10))

In [ ]:
from collections import Counter

# Contare i punti per ciascun cluster
counts = Counter(cluster_labels)

# Visualizzare i risultati
for cluster, count in counts.items():
    print(f"Cluster {cluster}: {count} punti")


In [ ]:
df['Cluster'] = cluster_labels
df.head()

In [ ]:
# print 5 vitims from each cluster
from tabulate import tabulate
def similar(i):
    similar = df.loc[df['Cluster'] == i][:10].reset_index()
    similar_print = similar[["name", "race", "manner_of_death", "age", "gender", "signs_of_mental_illness", "flee", "armed", "state"]]
    print("\nCluster ", i+1, "victims:")
    print(tabulate(similar_print, headers='keys', tablefmt='psql'))

for i in range(3):
    similar(i)

In [ ]:
cluster_data['Cluster'] = cluster_labels
cluster_data.head()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Creare una figura
plt.figure(figsize=(10, 8))

# Primo grafico
plt.subplot(2, 2, 1)
sns.countplot(x='threat_level', hue='Cluster', data=df, palette='viridis')
plt.title('Distribuzione del livello di minaccia per i cluster')

# Secondo grafico
plt.subplot(2, 2, 2)
sns.countplot(x='race', hue='Cluster', data=df, palette='viridis')
plt.title('Distribuzione della razza per i cluster')

# Terzo grafico
plt.subplot(2, 2, 3)
sns.countplot(x='AgeRange', hue='Cluster', data=cluster_data, palette='viridis')
plt.title('Distribuzione del range di età per i cluster')

# Quarto grafico
plt.subplot(2, 2, 4)
sns.countplot(x='flee', hue='Cluster', data=df, palette='viridis')
plt.title('Distribuzione di "flee" per i cluster')

# Mostrare i grafici
plt.tight_layout()
plt.show()




ASSOCIATION ANALYSIS

In [ ]:
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

In [ ]:
#Filter the transactions for cluster 0
cluster_0 = scaled_cluster_data[scaled_cluster_data['Cluster'] == 0].drop('Cluster',axis=1)
# Generate frequent itemsets using Apriori algorithm
frequent_itemsets_0 = apriori(cluster_0, min_support=0.015,use_colnames=True)
# Generate association rules
rules_0 = association_rules(frequent_itemsets_0, metric="lift", min_threshold=1)
# Sort the rules by lift in descending order
rules_0 = rules_0.sort_values(by='lift', ascending=False)

# Print the top 10 association rules
rules_0.head(10)

In [ ]:
r0 = rules_0[ (rules_0['lift'] >= 5) & (rules_0['confidence'] >= 0.6) ]
r0

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Generate scatterplot using support and confidence
sns.scatterplot(x = "support", y = "confidence",size = "lift", data = r0)
plt.show()

In [ ]:
#Filter the transactions for cluster 1
cluster_1 = scaled_cluster_data[scaled_cluster_data['Cluster'] == 1].drop('Cluster',axis=1)
# Generate frequent itemsets using Apriori algorithm
frequent_itemsets_1 = apriori(cluster_1, min_support=0.015,use_colnames=True)
# Generate association rules
rules_1 = association_rules(frequent_itemsets_1, metric="lift", min_threshold=1)
# Sort the rules by lift in descending order
rules_1 = rules_1.sort_values(by='lift', ascending=False)

# Print the top 10 association rules
rules_1.head(10)

In [ ]:
r1 = rules_1[ (rules_1['lift'] >= 15) & (rules_1['confidence'] >= 0.8) ]
r1

In [ ]:
# Generate scatterplot using support and confidence
sns.scatterplot(x = "support", y = "confidence",size = "lift", data = r1)
plt.show()

In [ ]:
#Filter the transactions for cluster 2
cluster_2 = scaled_cluster_data[scaled_cluster_data['Cluster'] == 2].drop('Cluster',axis=1)
# Generate frequent itemsets using Apriori algorithm
frequent_itemsets_2 = apriori(cluster_2, min_support=0.015,use_colnames=True)
# Generate association rules
rules_2 = association_rules(frequent_itemsets_2, metric="lift", min_threshold=1)
# Sort the rules by lift in descending order
rules_2 = rules_2.sort_values(by='lift', ascending=False)

# Print the top 10 association rules
rules_2.head(10)

In [ ]:
r2 = rules_2[ (rules_2['lift'] >= 5) & (rules_2['confidence'] >= 0.9) ]
r2

In [ ]:
# Generate scatterplot using support and confidence
sns.scatterplot(x = "support", y = "confidence",size = "lift", data = r2)
plt.show()

In [ ]:
data.head()

In [ ]:
columns_to_drop = ['name', 'id', 'longitude', 'latitude']

# Creazione di un nuovo DataFrame 'cluster_data' eliminando le colonne specificate
freq_data = data.drop(columns=columns_to_drop)


In [ ]:
#create categorical category for age 
bins = [0, 2, 18, 35, 65, np.inf]
names = ['<2', '2-18', '18-35', '35-65', '65+']

freq_data['AgeRange'] = pd.cut(freq_data['age'], bins, labels=names)
freq_data.head()

In [ ]:
#remove age columns
freq_data.drop('age', axis = 1, inplace = True)

In [ ]:
##convert categorial variables to dummy vars to use kmeans algotithm 
scaled_freq_data = pd.get_dummies(data=freq_data, columns=['manner_of_death', 'armed', 'gender', 'race', 'signs_of_mental_illness', 
                                   'threat_level', 'flee', 'body_camera', 'AgeRange', 'state', 'city'])

In [ ]:
scaled_freq_data = scaled_freq_data.astype('int64')
scaled_freq_data.head()

In [ ]:

# Generate frequent itemsets using Apriori algorithm
frequent_itemsets = apriori(scaled_freq_data, min_support=0.015,use_colnames=True)
# Generate association rules
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)
# Sort the rules by lift in descending order
rules = rules.sort_values(by='lift', ascending=False)

# Print the top 10 association rules
rules.head(10)

In [ ]:
r = rules[ (rules['lift'] >= 5) & (rules['confidence'] >= 0.7) ]
r

In [ ]:
# Generate scatterplot using support and confidence
sns.scatterplot(x = "support", y = "confidence",size = "lift", data = r)
plt.show()